# Multi-generator diagnostics without overclaiming

**Current surface:** V0.28.

## Purpose

Show the V0.27 multi-generator diagnostic surface: supplied `GeneratorFamily` objects, span/rank/closure diagnostics, candidate validation behavior, and a clear public-promotion decision.

## What you will learn

- How algebraic diagnostics differ from PDE-context validation.
- How closed, non-closed, and rank-deficient families are reported.
- Why closure does not imply residual symmetry.
- Why public multi-generator fitting, finite flows, BCH composition, and orbit charts remain deferred.

## Required extras

Core install is enough; Matplotlib is used for tutorial status plots.

## Expected runtime

Less than 1 minute.

## Out of scope

No public multi-generator fitting API, no finite multi-generator group action, no BCH composition, no multi-parameter orbit atlas, no learned symmetry-group training.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from notebooks._tutorial_utils import plot_label_strip, plot_named_metrics, pretty_json
from pdelie.examples import run_multi_generator_diagnostics_example

CONFIG = {"show_full_closure_reports": False}
CONFIG


## 1. Run the public diagnostic example

The example uses supplied generator families only. It does not fit a multi-generator family from PDE data.


In [ ]:
example = run_multi_generator_diagnostics_example()
plot_label_strip(
    {
        "release decision": example["decision"]["conclusion"],
        "fit probe": example["fit_probe_diagnostics"]["label"],
        "confidence": example["confidence"]["confidence_label"],
    },
    title="Multi-generator decision dashboard",
)
print(pretty_json(example["decision"], max_chars=2500))


## 2. Algebraic diagnostics are their own lane

A closed affine family, a non-closed polynomial family, and a rank-deficient family produce different diagnostics. None of these statements, by itself, says the PDE residual is preserved.


In [ ]:
algebra = example["algebraic_diagnostics"]
metrics = {
    "closed affine closure": algebra["closed_affine_x"]["closure"]["summary"],
    "nonclosed closure": algebra["nonclosed_polynomial"]["closure"]["summary"],
}
plot_named_metrics(metrics, title="Closure residual summaries")
print(pretty_json({
    "closed_affine": {
        "rank": algebra["closed_affine_x"]["family_rank_status"],
        "bracket_convention": algebra["closed_affine_x"]["bracket_convention"],
        "structure_constant_error": algebra["closed_affine_x"]["structure_constants"]["structure_constant_error"],
    },
    "nonclosed_polynomial": {
        "rank": algebra["nonclosed_polynomial"]["family_rank_status"],
        "closure_summary": algebra["nonclosed_polynomial"]["closure"]["summary"],
    },
    "rank_deficient": {
        "rank": algebra["rank_deficient_affine"]["family_rank_status"],
        "structure_constant_status": algebra["rank_deficient_affine"]["structure_constants"]["status"],
        "span_comparison_status": algebra["rank_deficient_span_comparison"]["comparison_status"],
    },
}, max_chars=4500))


## 3. PDE-context validation is a separate lane

Candidate validation can report closure evidence, span evidence, and configured residual/finite-transform checks. In V0.28, multi-row families with only algebraic evidence conclude at most `partially_validated`.


In [ ]:
pde_context = example["pde_context_diagnostics"]
plot_label_strip(
    {name: report["conclusion"] for name, report in pde_context.items()},
    title="Candidate validation conclusions",
)
print(pretty_json({
    name: {
        "candidate_kind": report["candidate_kind"],
        "conclusion": report["conclusion"],
        "source_candidate_id": report["source_candidate_id"],
    }
    for name, report in pde_context.items()
}, max_chars=4500))


## 4. The boundary is the feature

The report is designed to be useful to external or learned methods that can supply candidate generator families. PDELie diagnoses those families; it does not claim to learn multi-generator groups or construct finite augmentation orbits.


In [ ]:
print(pretty_json({
    "generator_source": example["extra_metrics"]["generator_source"],
    "closure_does_not_imply_pde_symmetry": example["extra_metrics"]["closure_does_not_imply_pde_symmetry"],
    "no_bch_composition": example["extra_metrics"]["no_bch_composition"],
    "no_exponential_flow_integration": example["extra_metrics"]["no_exponential_flow_integration"],
    "no_multi_parameter_orbit_chart": example["extra_metrics"]["no_multi_parameter_orbit_chart"],
}, max_chars=2500))


## Recap

V0.28 can diagnose supplied multi-generator families algebraically and empirically, while public multi-generator fitting and invariant-action machinery remain deferred.

## Common pitfalls

- Treating closure as PDE residual invariance.
- Treating rank deficiency as malformed input when the schema is valid.
- Using a diagnostic example as a fitting workflow.
- Assuming finite flows, BCH composition, or orbit charts exist for arbitrary families.

## Extension ideas

- Swap in an external method's supplied `GeneratorFamily` and compare closure/span reports.
- Use `closure_required=False` when non-closure should be a warning rather than a rejection.
- Pair algebra diagnostics with `validate_symmetry_candidate(...)` on a real residual context.

## What to read/run next

Run `07_external_symmetry_candidates.ipynb` for single-candidate interop, or `10_scope_decisions_and_weak_supportability.ipynb` for careful release-decision reporting.
